# Look at RACMO climate for the ERA5 reanalysis data

1. How stable is the climate for two spinup periods?
2. What is the average change in precipitation, surface temperature, and snowmelt over the whole period?

In [1]:
import xarray as xr
import glob
import numpy as np
import matplotlib.pyplot as plt


In [2]:
#set paths
variables = ["ff10m","precip","evap","sndiv","snowfall","snowmelt","tskin"]
RACMO_ts_dir = "/home/nld4814/scratch/FGRN055_era055/input/timeseries"
RACMO_avg_dir = "/home/nld4814/scratch/FGRN055_era055/input/averages"

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

MASK_FILE = "/home/nld4814/perm/code/IMAU-FDM/reference/FGRN055/FGRN055_Masks.nc"

# --- Load ice mask as a plain numpy array (566, 438) ---
ds_mask = xr.open_dataset(MASK_FILE)
icemask_np = ds_mask["Icemask_GR"].squeeze(drop=True).values.astype(bool)
ds_mask.close()
print(f"Ice-sheet cells: {icemask_np.sum()}")

# Get rlon reference values from one file to map file columns → mask columns
ref_file = sorted(glob.glob(f"{RACMO_ts_dir}/{variables[0]}_*.nc"))[0]
with xr.open_dataset(ref_file) as ds_ref:
    rlon_step  = float(np.diff(ds_ref.rlon.values).mean())  # ~0.05°
    rlon_start = float(ds_ref.rlon.values[0])               # smallest rlon in domain

print(f"rlon start: {rlon_start:.4f}°, step: {rlon_step:.4f}°")


def load_domain_mean(var, ts_dir, mask_np=None, time_slice=slice("1940", "1970"),
                     resample_freq="ME"):
    """
    Process one longitude-band file at a time (no dask):
      1. Open file, slice to time period
      2. Resample to monthly means  →  (n_months, 566, 6)  [small]
      3. Apply ice mask columns for this band, accumulate weighted sum
    Returns an xarray DataArray of monthly ice-sheet-mean values.
    """
    files = sorted(glob.glob(f"{ts_dir}/{var}_*.nc"))

    running_sum = None
    n_ice_total = 0
    time_coord  = None

    for f in files:
        with xr.open_dataset(f) as ds:
            da = ds[var].squeeze("height", drop=True).sel(time=time_slice)

            # Resample BEFORE spatial ops — reduces 87k → ~360 monthly steps
            da_monthly = da.resample(time=resample_freq).mean()  # (n_months, 566, n_rlon)

            if mask_np is not None:
                col_idx   = np.round((ds.rlon.values - rlon_start) / rlon_step).astype(int)
                file_mask = mask_np[:, col_idx]                  # (566, n_rlon)
                n_valid   = int(file_mask.sum())
                band_sum  = (da_monthly.values * file_mask[np.newaxis]).sum(axis=(1, 2))
            else:
                n_valid  = da_monthly.shape[1] * da_monthly.shape[2]
                band_sum = da_monthly.values.sum(axis=(1, 2))

            n_ice_total += n_valid

            if running_sum is None:
                running_sum = band_sum
                time_coord  = da_monthly.time.values
            else:
                running_sum += band_sum

    domain_mean = running_sum / n_ice_total
    return xr.DataArray(domain_mean, coords={"time": time_coord}, dims=["time"])


# --- Load all variables ---
print("\nLoading variables — processing one longitude band at a time...")
data = {}
for var in variables:
    print(f"  {var}...", end=" ", flush=True)
    data[var] = load_domain_mean(var, RACMO_ts_dir, mask_np=icemask_np)
    print("done")
print("All variables loaded.")

In [ ]:
var_meta = {
    "ff10m":    ("Wind speed at 10 m",  "m s⁻¹"),
    "precip":   ("Precipitation",       "kg m⁻² s⁻¹"),
    "evap":     ("Evaporation",         "kg m⁻² s⁻¹"),
    "sndiv":    ("Snow drifting",       "kg m⁻² s⁻¹"),
    "snowfall": ("Snowfall",            "kg m⁻² s⁻¹"),
    "snowmelt": ("Snowmelt",            "kg m⁻² s⁻¹"),
    "tskin":    ("Skin temperature",    "K"),
}

ncols = 3
nrows = (len(variables) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), sharex=True)

for i, var in enumerate(variables):
    ax = axes.flat[i]
    t  = data[var].time.values
    # Decimal year for trend fitting and x-axis
    t_yr = data[var].time.dt.year.values + (data[var].time.dt.month.values - 1) / 12
    vals = data[var].values

    ax.plot(t_yr, vals, lw=0.8, color="steelblue", alpha=0.7)

    # Annual rolling mean for readability
    annual = data[var].resample(time="YE").mean()
    t_ann  = annual.time.dt.year.values + 0.5   # mid-year
    ax.plot(t_ann, annual.values, lw=2, color="navy", label="annual mean")

    # Linear trend on monthly values
    slope, intercept = np.polyfit(t_yr, vals, 1)
    ax.plot(t_yr, slope * t_yr + intercept, "r--", lw=1.2,
            label=f"trend: {slope * 10:.3g} / decade")

    label, unit = var_meta[var]
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(unit, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for j in range(len(variables), len(axes.flat)):
    axes.flat[j].set_visible(False)

fig.suptitle(
    "RACMO ERA5 — FGRN055 ice-sheet-mean monthly climate (1940–1970 spinup period)",
    fontsize=13,
)
fig.supxlabel("Year", fontsize=10)
plt.tight_layout()
plt.show()